# 03 — OOF Assembly, Ensemble Decision, Calibration

Dijalankan setelah semua 20 model fold (4 arsitektur x 5 fold) selesai mengekspor prediksi OOF.
CPU-only sudah cukup. Nol akses data test (PRD §2 rule 3) — notebook ini hanya pernah menyentuh CSV OOF.

Urutan tetap (PRD §10.3, tidak boleh diubah): softmax mentah per model -> fold-average per
arsitektur (concat OOF) -> weighted soft-vote antar arsitektur (§10.2) -> SATU kalibrasi (§10.3).
Tidak ada kalibrasi per-model, tidak ada kalibrasi sebelum bobot ensemble final ditentukan.


In [ ]:
# scipy/scikit-learn/pandas/numpy sudah terpasang di image Kaggle/Colab (dipakai puluhan paket
# bawaan lain) — tidak di-pin exact di sini supaya tidak bentrok dengan resolver pip platform
# (deviasi dari PRD §12; lihat catatan yang sama di NB01/02x). Versi yang aktif dipakai tercetak
# di sel berikutnya untuk audit trail.
!pip install -q scipy


In [ ]:
import json
import hashlib
import sklearn
import scipy
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import f1_score

SEED = 42
np.random.seed(SEED)
print("numpy:", np.__version__, "| pandas:", pd.__version__, "| scipy:", scipy.__version__, "| scikit-learn:", sklearn.__version__)

OOF_ROOT = Path("/kaggle/working/oof")  # atau lokasi lain tempat CSV OOF dari 4 anggota dikumpulkan
FOLD_ASSIGNMENT_PATH = Path("/kaggle/input/bdc2026-master-data/manifests/fold_assignment.csv")
OUTPUT_ROOT = Path("/kaggle/working")

ARCHS = ["convnextv2_base", "swinv2_base_window12to24", "maxvit_base_tf", "tf_efficientnetv2_m"]
N_FOLDS = 5
NUM_CLASSES = 3

# Gate review manual untuk guardrail 3 (PRD 10.2). Biarkan False sampai ada yang mereview
# kontribusi arsitektur yang nyaris nol dan menyetujuinya secara eksplisit di oof_report.md.
MANUAL_REVIEW_OVERRIDE_LOW_WEIGHT = False


## §10.1 Assersi integritas fail-fast (jalankan sebelum apa pun yang lain)


In [ ]:
def file_md5(path, chunk_size=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


fold_assignment = pd.read_csv(FOLD_ASSIGNMENT_PATH)
expected_ids = set(fold_assignment["image_id"])

arch_oof = {}  # arch -> DataFrame terurut berdasarkan image_id
provenance_stamps = []

for arch in ARCHS:
    fold_frames = []
    for k in range(N_FOLDS):
        csv_path = OOF_ROOT / f"oof_{arch}_fold{k}.csv"
        json_path = OOF_ROOT / f"oof_{arch}_fold{k}.json"
        assert csv_path.exists(), f"file OOF hilang: {csv_path}"
        assert json_path.exists(), f"sidecar provenance hilang: {json_path}"
        df = pd.read_csv(csv_path)
        assert set(df["fold"].unique()) == {k}, f"{csv_path} berisi baris di luar fold {k}"
        fold_frames.append(df)
        with open(json_path) as f:
            provenance_stamps.append({"arch": arch, "fold": k, **json.load(f)})

    concat = pd.concat(fold_frames, ignore_index=True)
    assert concat["image_id"].is_unique, f"{arch}: image_id duplikat lintas file OOF fold"
    assert set(concat["image_id"]) == expected_ids, f"{arch}: set id OOF != set id fold_assignment.csv"
    arch_oof[arch] = concat.sort_values("image_id").reset_index(drop=True)
    print(f"{arch}: {len(concat)} baris OOF, set id cocok dengan fold_assignment.csv")

# setiap arsitektur wajib mencakup set id yang sama persis, dengan urutan yang sama setelah disortir
first_ids = arch_oof[ARCHS[0]]["image_id"].values
for arch in ARCHS[1:]:
    assert np.array_equal(arch_oof[arch]["image_id"].values, first_ids), f"{arch}: urutan image_id tidak cocok"

# semua stempel provenance §5.2 wajib cocok di keempat arsitektur (guard artefak basi)
stamp_keys = ["fold_assignment_md5", "master_dataset_version", "skeleton_version"]
reference_stamp = {k: provenance_stamps[0][k] for k in stamp_keys}
for s in provenance_stamps:
    for k in stamp_keys:
        assert s[k] == reference_stamp[k], (
            f"provenance tidak cocok pada {k}: {s['arch']} fold{s['fold']} punya {s[k]!r}, "
            f"seharusnya {reference_stamp[k]!r} — ada anggota yang pakai artefak basi, stop dan perbaiki dulu."
        )
print("Semua assersi integritas fail-fast NB03 lolos.")
print("stempel provenance referensi:", reference_stamp)


In [ ]:
y_true = arch_oof[ARCHS[0]]["true_label"].values
for arch in ARCHS[1:]:
    assert np.array_equal(arch_oof[arch]["true_label"].values, y_true), f"{arch}: true_label tidak cocok dengan {ARCHS[0]}"

prob_cols = ["prob_0", "prob_1", "prob_2"]
arch_probs = {arch: arch_oof[arch][prob_cols].values for arch in ARCHS}  # softmax mentah, tergabung per fold per arsitektur
oof_fold = arch_oof[ARCHS[0]]["fold"].values


## §10.2 Konstruksi ensemble — guarded weighted average

`w = softmax(z)`, `z` dioptimasi via Nelder-Mead untuk memaksimalkan OOF Macro-F1 ensemble.
Nelder-Mead multi-start (5 restart) mengurangi risiko terjebak di local optimum yang buruk;
metodenya sendiri tidak berubah.


In [ ]:
def softmax_np(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def ensemble_probs(weights, probs_by_arch):
    return sum(w * p for w, p in zip(weights, probs_by_arch))


def neg_macro_f1_for_z(z, probs_by_arch, y):
    w = softmax_np(z)
    preds = ensemble_probs(w, probs_by_arch).argmax(axis=1)
    return -f1_score(y, preds, average="macro")


def search_weights(probs_by_arch, y, n_restarts=5, seed=SEED):
    rng = np.random.default_rng(seed)
    n_arch = len(probs_by_arch)
    best_res = None
    for i in range(n_restarts):
        z0 = np.zeros(n_arch) if i == 0 else rng.normal(scale=0.5, size=n_arch)
        res = minimize(neg_macro_f1_for_z, z0, args=(probs_by_arch, y), method="Nelder-Mead",
                        options={"xatol": 1e-4, "fatol": 1e-6, "maxiter": 2000})
        if best_res is None or res.fun < best_res.fun:
            best_res = res
    return softmax_np(best_res.x), -best_res.fun


probs_by_arch_full = [arch_probs[a] for a in ARCHS]
optimized_weights, optimized_f1 = search_weights(probs_by_arch_full, y_true)
equal_weights = np.full(len(ARCHS), 1.0 / len(ARCHS))
equal_f1 = f1_score(y_true, ensemble_probs(equal_weights, probs_by_arch_full).argmax(axis=1), average="macro")

print("bobot optimized:", dict(zip(ARCHS, optimized_weights.round(4))), "-> OOF macro-F1 =", round(optimized_f1, 5))
print("bobot equal:    ", dict(zip(ARCHS, equal_weights.round(4))), "-> OOF macro-F1 =", round(equal_f1, 5))


### Guardrail 1 — stability check (leave-one-fold-out)

Ulangi pencarian bobot di 5 subset OOF leave-one-fold-out; tolak kalau ada `w_i` dengan koefisien
variasi > 0.3 di antara 5 hasil refit.


In [ ]:
loFo_weights = []
for k in range(N_FOLDS):
    mask = oof_fold != k
    probs_subset = [p[mask] for p in probs_by_arch_full]
    y_subset = y_true[mask]
    w_k, _ = search_weights(probs_subset, y_subset, n_restarts=3, seed=SEED + k)
    loFo_weights.append(w_k)

loFo_weights = np.array(loFo_weights)  # (5, n_arch)
cv_per_arch = loFo_weights.std(axis=0) / np.clip(loFo_weights.mean(axis=0), 1e-8, None)
guardrail_1_pass = bool((cv_per_arch <= 0.3).all())
print("CV bobot leave-one-fold-out per arsitektur:", dict(zip(ARCHS, cv_per_arch.round(4))))
print("guardrail 1 (stability):", "LOLOS" if guardrail_1_pass else "GAGAL")


### Guardrail 2 — minimum-gain floor

OOF Macro-F1 bobot optimized wajib melampaui OOF Macro-F1 bobot equal dengan selisih >= 0.002 absolut.


In [ ]:
guardrail_2_pass = bool((optimized_f1 - equal_f1) >= 0.002)
print(f"gain = {optimized_f1 - equal_f1:.5f}")
print("guardrail 2 (minimum-gain floor):", "LOLOS" if guardrail_2_pass else "GAGAL")


### Guardrail 3 — per-architecture floor

Tidak ada `w_i` yang boleh dioptimasi di bawah 0.05 tanpa review manual yang dicatat di `oof_report.md`.


In [ ]:
low_weight_archs = [a for a, w in zip(ARCHS, optimized_weights) if w < 0.05]
guardrail_3_pass = bool(len(low_weight_archs) == 0 or MANUAL_REVIEW_OVERRIDE_LOW_WEIGHT)
if low_weight_archs and not MANUAL_REVIEW_OVERRIDE_LOW_WEIGHT:
    print(f"guardrail 3: {low_weight_archs} dioptimasi di bawah 0.05 — butuh review manual "
          f"(set MANUAL_REVIEW_OVERRIDE_LOW_WEIGHT=True di atas setelah direview, lalu jalankan ulang sel ini)")
print("guardrail 3 (per-architecture floor):", "LOLOS" if guardrail_3_pass else "GAGAL")


In [ ]:
all_guardrails_pass = guardrail_1_pass and guardrail_2_pass and guardrail_3_pass
if all_guardrails_pass:
    final_weights, final_scheme, final_f1 = optimized_weights, "optimized", optimized_f1
else:
    final_weights, final_scheme, final_f1 = equal_weights, "equal", equal_f1
print(f"Skema ensemble FINAL: {final_scheme} (all_guardrails_pass={all_guardrails_pass})")
print("bobot final:", dict(zip(ARCHS, final_weights.round(4))))


### Ablasi leave-one-architecture-out (laporan transparansi)

Menormalkan ulang vektor bobot final atas 3 arsitektur yang tersisa untuk tiap arsitektur yang
dihilangkan, menunjukkan kontribusi marjinal masing-masing arsitektur.


In [ ]:
leave_one_arch_out = {}
for i, held_out in enumerate(ARCHS):
    remaining_idx = [j for j in range(len(ARCHS)) if j != i]
    w_remaining = final_weights[remaining_idx]
    w_remaining = w_remaining / w_remaining.sum()
    probs_remaining = [probs_by_arch_full[j] for j in remaining_idx]
    f1_without = f1_score(y_true, ensemble_probs(w_remaining, probs_remaining).argmax(axis=1), average="macro")
    leave_one_arch_out[held_out] = {"oof_macro_f1_without_this_arch": float(f1_without),
                                     "delta_vs_full_ensemble": float(final_f1 - f1_without)}
    print(f"tanpa {held_out}: OOF macro-F1 = {f1_without:.5f} (ensemble penuh - ini = {final_f1 - f1_without:+.5f})")


## §10.3 Kalibrasi = post-hoc logit adjustment (diterapkan SEKALI, di atas ensemble yang sudah diberi bobot)

`argmax_c[log(p_ens,c) + delta_c]`, `delta_0 := 0`; grid kasar (step 0.05) untuk `delta_1, delta_2
in [-1.5, 1.5]`, lalu refinement lokal via Nelder-Mead. Aturan konservatisme: di antara solusi yang
selisihnya <= 0.1% absolut dari F1 OOF maksimum, pilih `||delta||` terkecil.


In [ ]:
ens_probs_final = ensemble_probs(final_weights, probs_by_arch_full)


def apply_logit_adjustment(probs, delta):
    log_probs = np.log(np.clip(probs, 1e-12, 1.0))
    return (log_probs + delta).argmax(axis=1)


def macro_f1_for_delta(delta12, probs, y):
    delta = np.array([0.0, delta12[0], delta12[1]])
    preds = apply_logit_adjustment(probs, delta)
    return f1_score(y, preds, average="macro")


def search_calibration(probs, y, grid_step=0.05, bound=1.5, conservatism_tol=0.001):
    grid = np.round(np.arange(-bound, bound + 1e-9, grid_step), 4)
    candidates = []
    for d1 in grid:
        for d2 in grid:
            f1 = macro_f1_for_delta((d1, d2), probs, y)
            candidates.append((f1, float(np.hypot(d1, d2)), (float(d1), float(d2))))

    coarse_best = max(candidates, key=lambda c: c[0])
    refine_res = minimize(lambda d: -macro_f1_for_delta(d, probs, y), x0=np.array(coarse_best[2]),
                           method="Nelder-Mead", options={"xatol": 1e-3, "fatol": 1e-6})
    refined_delta = tuple(np.clip(refine_res.x, -bound, bound).tolist())
    refined_f1 = macro_f1_for_delta(refined_delta, probs, y)
    candidates.append((refined_f1, float(np.hypot(*refined_delta)), refined_delta))

    max_f1 = max(c[0] for c in candidates)
    near_max = [c for c in candidates if c[0] >= max_f1 - conservatism_tol]
    near_max.sort(key=lambda c: c[1])  # ||delta|| terkecil di antara solusi near-max
    chosen_f1, chosen_norm, chosen_delta = near_max[0]
    return np.array([0.0, chosen_delta[0], chosen_delta[1]]), chosen_f1


delta, f1_after_calibration = search_calibration(ens_probs_final, y_true)
f1_before_calibration = f1_score(y_true, ens_probs_final.argmax(axis=1), average="macro")
print(f"OOF macro-F1 sebelum kalibrasi: {f1_before_calibration:.5f}")
print(f"OOF macro-F1 sesudah kalibrasi: {f1_after_calibration:.5f}  (delta={delta.round(4).tolist()})")


In [ ]:
y_pred_calibrated = apply_logit_adjustment(ens_probs_final, delta)
per_class_f1_after = f1_score(y_true, y_pred_calibrated, average=None, labels=[0, 1, 2]).tolist()
pred_dist = pd.Series(y_pred_calibrated).value_counts(normalize=True).sort_index().to_dict()

calibration_params = {
    "delta": delta.tolist(),
    "method": "grid(step=0.05, bound=1.5)+nelder_mead_refine; conservatism_tol=0.001 -> min ||delta||",
    "oof_macro_f1_before": float(f1_before_calibration),
    "oof_macro_f1_after": float(f1_after_calibration),
    "per_class_f1_after": per_class_f1_after,
    "predicted_class_distribution": {int(k): float(v) for k, v in pred_dist.items()},
}
with open(OUTPUT_ROOT / "calibration_params.json", "w") as f:
    json.dump(calibration_params, f, indent=2)
print(json.dumps(calibration_params, indent=2))


## Output — `ensemble_config.json` dan `oof_report.md`


In [ ]:
ensemble_config = {
    "archs": ARCHS,
    "scheme_adopted": final_scheme,
    "optimized_weights": dict(zip(ARCHS, optimized_weights.tolist())),
    "equal_weights": dict(zip(ARCHS, equal_weights.tolist())),
    "final_weights": dict(zip(ARCHS, final_weights.tolist())),
    "guardrails": {
        "stability_cv_per_arch": dict(zip(ARCHS, cv_per_arch.tolist())),
        "stability_pass": guardrail_1_pass,
        "minimum_gain": float(optimized_f1 - equal_f1),
        "minimum_gain_pass": guardrail_2_pass,
        "low_weight_archs": low_weight_archs,
        "manual_review_override_used": MANUAL_REVIEW_OVERRIDE_LOW_WEIGHT,
        "per_arch_floor_pass": guardrail_3_pass,
        "all_pass": all_guardrails_pass,
    },
    "oof_macro_f1_optimized_weights": float(optimized_f1),
    "oof_macro_f1_equal_weights": float(equal_f1),
    "reference_provenance": reference_stamp,
}
with open(OUTPUT_ROOT / "ensemble_config.json", "w") as f:
    json.dump(ensemble_config, f, indent=2)

report_lines = [
    "# Laporan OOF NB03\n",
    f"Provenance referensi: `{reference_stamp}`\n",
    "## Pencarian bobot ensemble\n",
    f"- OOF Macro-F1 bobot equal: **{equal_f1:.5f}**",
    f"- OOF Macro-F1 bobot optimized: **{optimized_f1:.5f}** (gain {optimized_f1 - equal_f1:+.5f})",
    f"- Skema yang dipakai: **{final_scheme}** (all_guardrails_pass={all_guardrails_pass})\n",
    "| arsitektur | w optimized | w equal | w final |",
    "|---|---|---|---|",
] + [
    f"| {a} | {optimized_weights[i]:.4f} | {equal_weights[i]:.4f} | {final_weights[i]:.4f} |"
    for i, a in enumerate(ARCHS)
] + [
    "\n## Guardrail\n",
    f"1. Stability (CV leave-one-fold-out <= 0.3): {'LOLOS' if guardrail_1_pass else 'GAGAL'} — "
    f"{dict(zip(ARCHS, cv_per_arch.round(4)))}",
    f"2. Minimum-gain floor (>= 0.002 absolut): {'LOLOS' if guardrail_2_pass else 'GAGAL'} — "
    f"gain={optimized_f1 - equal_f1:.5f}",
    f"3. Per-architecture floor (>= 0.05 atau review manual): {'LOLOS' if guardrail_3_pass else 'GAGAL'} — "
    f"low_weight_archs={low_weight_archs}, manual_review_override={MANUAL_REVIEW_OVERRIDE_LOW_WEIGHT}\n",
    "## Ablasi leave-one-architecture-out\n",
    "| arsitektur dihilangkan | OOF macro-F1 tanpanya | delta vs ensemble penuh |",
    "|---|---|---|",
] + [
    f"| {a} | {v['oof_macro_f1_without_this_arch']:.5f} | {v['delta_vs_full_ensemble']:+.5f} |"
    for a, v in leave_one_arch_out.items()
] + [
    "\n## Kalibrasi (logit adjustment)\n",
    f"- delta: {delta.round(4).tolist()}",
    f"- OOF macro-F1 sebelum kalibrasi: {f1_before_calibration:.5f}",
    f"- OOF macro-F1 sesudah kalibrasi: {f1_after_calibration:.5f}",
    f"- Per-class F1 sesudah kalibrasi: {[round(x, 4) for x in per_class_f1_after]}",
    f"- Distribusi kelas prediksi sesudah kalibrasi: {pred_dist}",
]
with open(OUTPUT_ROOT / "oof_report.md", "w") as f:
    f.write("\n".join(report_lines) + "\n")

print("Menulis ensemble_config.json, calibration_params.json, oof_report.md ke", OUTPUT_ROOT)
